## Import and Setup

In [2]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from pathlib import Path

Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('data/processed').mkdir(parents=True, exist_ok=True)
Path('reports/figures').mkdir(parents=True, exist_ok=True)

/Users/parisakalaki/data-engineering-foundations/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load datasets

In [3]:
print('Loading Counsel Chat...')
df_counsel = load_dataset('nbertagnolli/counsel-chat', split='train').to_pandas()
df_counsel.to_csv('../data/raw/counsel_chat_raw.csv', index=False)
print(f'✅ Counsel Chat: {len(df_counsel):,} rows')
print(f'Columns: {list(df_counsel.columns)}')

print('\nLoading Crisis dataset...')
df_crisis = pd.read_csv('../data/raw/crisis_raw.csv')
print(df_crisis.head(2))
print(df_crisis.columns.tolist())
print(df_crisis['class'].value_counts())

Loading Counsel Chat...


Repo card metadata block was not found. Setting CardData to empty.


✅ Counsel Chat: 2,775 rows
Columns: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views']

Loading Crisis dataset...
   Unnamed: 0                                               text        class
0           2  Ex Wife Threatening SuicideRecently I left my ...      suicide
1           3  Am I weird I don't get affected by compliments...  non-suicide
['Unnamed: 0', 'text', 'class']
class
suicide        116037
non-suicide    116037
Name: count, dtype: int64


In [4]:
print('=== COUNSEL CHAT ===')
print(df_counsel.head(2))
print(df_counsel.isnull().sum())

print('\n=== CRISIS DATASET ===')
print(df_crisis.head(2))
print(df_crisis.isnull().sum())
print(df_crisis.iloc[:, -1].value_counts())  # label distribution

=== COUNSEL CHAT ===
   questionID                              questionTitle  \
0           0  Do I have too many issues for counseling?   
1           0  Do I have too many issues for counseling?   

                                        questionText  \
0  I have so many issues to address. I have a his...   
1  I have so many issues to address. I have a his...   

                                        questionLink       topic  \
0  https://counselchat.com/questions/do-i-have-to...  depression   
1  https://counselchat.com/questions/do-i-have-to...  depression   

                                       therapistInfo  \
0  Jennifer MolinariHypnotherapist & Licensed Cou...   
1  Jason Lynch, MS, LMHC, LCAC, ADSIndividual & C...   

                                        therapistURL  \
0  https://counselchat.com/therapists/jennifer-mo...   
1  https://counselchat.com/therapists/jason-lynch...   

                                          answerText  upvotes  views  
0  It is very c

In [5]:
# Basic text cleaning function

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text

# Clean Counsel Chat data
df_counsel['questionText'] = df_counsel['questionText'].apply(clean_text)
df_counsel['answerText']   = df_counsel['answerText'].apply(clean_text)

# Drop short answers
df_counsel = df_counsel[df_counsel['answerText'].str.len() >= 50]
df_counsel = df_counsel.dropna(subset=['questionText', 'answerText'])

# Keep all therapist answers (duplicates are fine for RAG — more answers = richer knowledge base)
df_counsel['document'] = (
    'Question: ' + df_counsel['questionText'] +
    '\nAnswer: '  + df_counsel['answerText']
)

df_counsel[['questionText', 'answerText', 'topic', 'document']].to_csv(
    'data/processed/counsel_chat_clean.csv', index=False
)
print(f'✅ Counsel Chat saved: {len(df_counsel):,} rows')



# Clean Crisis dataset
# We know columns are 'text' and 'class'
df_crisis = df_crisis[['text', 'class']].copy()
df_crisis.columns = ['text', 'label']

df_crisis['text'] = df_crisis['text'].apply(clean_text)
df_crisis = df_crisis[df_crisis['text'].str.len() >= 20].dropna()

# We know labels are 'suicide' / 'non-suicide'
df_crisis['label_binary'] = df_crisis['label'].map(
    {'suicide': 1, 'non-suicide': 0}
)

print(f'✅ Crisis cleaned: {len(df_crisis):,} rows')
print(df_crisis['label_binary'].value_counts())

✅ Counsel Chat saved: 2,744 rows
✅ Crisis cleaned: 231,510 rows
label_binary
1    115857
0    115653
Name: count, dtype: int64


## Splitting

In [6]:
train, temp = train_test_split(df_crisis, test_size=0.30, random_state=42,
                                stratify=df_crisis['label_binary'])
val, test   = train_test_split(temp, test_size=0.50, random_state=42,
                                stratify=temp['label_binary'])

train.to_csv('data/processed/crisis_train.csv', index=False)
val.to_csv(  'data/processed/crisis_val.csv',   index=False)
test.to_csv( 'data/processed/crisis_test.csv',  index=False)

print(f'Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')

Train: 162,057 | Val: 34,726 | Test: 34,727
